In [1]:
import pandas as pd

In [2]:
df  = pd.read_csv("../data/my_scraped_data.csv")

In [3]:
df.columns

Index(['page_id', 'url', 'categories', 'sharh', 'hadith', 'rawy', 'mohadth',
       'source', 'page', 'hokm', 'takhrij'],
      dtype='object')

In [4]:
df["categories"].isna().sum()

np.int64(16)

In [5]:
df["categories"][187397]

'أذان - بدء الأذان ، رؤيا - التواطؤ على الرؤيا ، رؤيا - رؤيا الصالحين ، مناقب وفضائل - بلال بن رباح ، مناقب وفضائل – عبد الله بن زيد'

In [6]:
import pandas as pd


all_categories = (
    df["categories"]
    .dropna()
    .str.split(r"\s+،\s+")      # split on Arabic comma
    .explode()                  # one category per row
    .str.strip()
)

unique_categories = set(all_categories)

print(f"Unique categories: {len(unique_categories)}")

Unique categories: 7594


In [15]:
cat_df = (
    all_categories
    .str.split(r"\s*[-–]\s*", n=1, expand=True)
    .apply(lambda col: col.str.strip())
)

cat_df.columns = ["high_level", "low_level"]

In [22]:
cat_df["low_level"] = cat_df["low_level"].fillna("")

# Build category tree
category_tree = {}

for high, group in cat_df.groupby("high_level"):
    low_counts = (
        group.loc[group["low_level"] != "", "low_level"]
        .value_counts()
        .sort_values(ascending=False)
        .to_dict()
    )

    category_tree[high] = {
        "count": len(group),                  # Total items in this branch
        "num_children": len(low_counts),      # Number of unique low-level categories
        "children": low_counts                # {low_level: count}
    }

In [23]:
category_tree

{'آداب الدعاء': {'count': 4490,
  'num_children': 35,
  'children': {'استجابة الدعاء': 1160,
   'أوقات الإجابة': 625,
   'الثناء على الله في الدعاء': 447,
   'رفع اليدين في الدعاء': 367,
   'الاجتهاد في الدعاء': 196,
   'سؤال العبد ربه جميع حوائجه': 178,
   'الصلاة على النبي صلى الله عليه وسلم في الدعاء': 147,
   'قبول دعاء المسلم': 137,
   'لزوم الدعاء والإلحاح فيه': 129,
   'التأمين على الدعاء': 124,
   'الاعتداء في الدعاء': 118,
   'من يستجاب دعاؤهم': 114,
   'التضرع والتخشع والتمسكن في الدعاء': 96,
   'موانع إجابة الدعاء': 89,
   'الاستعجال في الدعاء والإجابة': 87,
   'بم يستفتح به الدعاء': 75,
   'الدعاء مستقبل القبلة': 61,
   'العزم في الدعاء': 41,
   'التوسل وأحكامه': 35,
   'بدء الداعي بنفسه': 34,
   'التوسل بصالح الأعمال في الدعاء': 30,
   'الزجر عن الإفراد بالدعاء': 30,
   'تكرير الدعاء': 28,
   'العجز في الدعاء': 23,
   'الدعاء بالأعمال الصالحة': 19,
   'كراهة الدعاء بتعجيل العقوبة في الدنيا': 17,
   'الدعاء بكف واحد': 16,
   'التعميم في الدعاء': 15,
   'رفع السبابة في الدعا

In [16]:
main_categories = pd.DataFrame(cat_df['high_level'].unique())

In [17]:
len(main_categories)

158

In [18]:
main_categories.columns = ['high_level_category']

In [19]:
main_categories.to_csv("../data/main_categories.csv")

In [20]:
pd.set_option("display.max_rows", None)
cat_df['high_level'].value_counts()

high_level
صلاة                                       63128
رقائق وزهد                                 53713
مناقب وفضائل                               47389
حج                                         39595
فضائل النبي وصفته ودلائل النبوة            37752
إيمان                                      35815
اعتصام بالسنة                              25312
أدعية وأذكار                               22329
صيام                                       21687
علم                                        19353
نكاح                                       19266
جهاد                                       18512
آداب عامة                                  17488
أطعمة                                      16739
وضوء                                       15798
بيوع                                       15483
بر وصلة                                    15476
صلاة الجماعة والإمامة                      14605
قرآن                                       13819
زينة اللباس                                13691
طهارة    

In [21]:

df[df['categories'].fillna("").str.contains("مناقب وفضائل – عبد الله بن زيد")]

,page_id,url,categories,sharh,hadith,rawy,mohadth,source,page,hokm,takhrij
36617,218055,https://dorar.net/hadith/sharh/218055,أضاحي - قسمة الإمام الأضاحي بين الناس ، زينة ا...,خص الله نبينا صلى الله عليه وسلم بخصائص دون سا...,- أنَّ محمَّدَ بنَ عبدِ اللَّهِ بنِ زيدٍ حدَّث...,محمد بن عبدالله بن زيد,الذهبي,تاريخ الإسلام,1/425,مرسل,أخرجه ابن خزيمة (2931)، والحاكم (1744)، والبيه...
60121,72406,https://dorar.net/hadith/sharh/72406,أذان - بدء الأذان ، رؤيا - التواطؤ على الرؤيا ...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- اهتمَّ النبيُّ صلَّى اللهُ عليْهِ وسلَّمَ لل...,عمومة أبي عمير بن أنس,ابن حجر العسقلاني,فتح الباري لابن حجر,2/97,إسناده صحيح,أخرجه أبو داود (498)، والبيهقي (1908) باختلاف ...
107482,29931,https://dorar.net/hadith/sharh/29931,جهاد - الرايات والألوية ، أذان - بدء الأذان ، ...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- اهتمَّ النبيُّ صلَّى اللهُ عليهِ وسلَّمَ للص...,عمومة أبي عمير بن أنس,الألباني,صحيح أبي داود,498,حسن,أخرجه أبو داود (498) واللفظ له، والبيهقي (1908...
107485,29934,https://dorar.net/hadith/sharh/29934,أذان - ألفاظ الأذان ، أذان - بدء الأذان ، رؤيا...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- كانَ رسولُ اللَّهِ صلَّى اللَّهُ عليهِ وسلَّ...,عبدالله بن زيد,الألباني,صحيح ابن ماجه,586,حسن,أخرجه أبو داود (499)، وابن ماجة (706)، وابن ال...
137452,65505,https://dorar.net/hadith/sharh/65505,أذان - بدء الأذان ، رؤيا - الرؤيا الصالحة من ا...,في هذا الحديث يصف أبو رمثة رضي الله عنه بعض هي...,- أنَّ عبدَ اللَّهِ بنَ زيدٍ الأنصارِيَّ جاءَ ...,أصحاب النبي صلى الله عليه وسلم,ابن الملقن,تحفة المحتاج,1/269,إسناده على شرط الصحيح,أخرجه البيهقي (1998)، وابن أبي شيبة في ((مصنفه...
139354,67469,https://dorar.net/hadith/sharh/67469,جهاد - الرايات والألوية ، أذان - بدء الأذان ، ...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- اهتمّ النبيُّ صلى الله عليه وسلم للصلاةِ كيف...,عمومة أبي عمير بن أنس,ابن عبدالبر,التمهيد,24/21,حسن,أخرجه أبو داود (498) واللفظ له، والبيهقي (1908...
140204,68344,https://dorar.net/hadith/sharh/68344,جهاد - الرايات والألوية ، أذان - بدء الأذان ، ...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- اهتمَّ النَّبيُّ صلَّى اللهُ علَيهِ وسلَّمَ ...,عمومة أبي عمير بن أنس,الألباني,جلباب المرأة,167,صحيح,أخرجه أبو داود (498)، والبيهقي (1908) باختلاف ...
143974,142270,https://dorar.net/hadith/sharh/142270,أذان - بدء الأذان ، مناقب وفضائل – عبد الله بن...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,- من رُؤْيا عبدِ اللهِ بنِ زيدٍ، وأنَّ عُمَرَ ...,NaN,القاضي عياض,إكمال المعلم,2/238,صحيح,NaN
172501,218055,https://dorar.net/hadith/sharh/218055,أضاحي - قسمة الإمام الأضاحي بين الناس ، زينة ا...,خص الله نبينا صلى الله عليه وسلم بخصائص دون سا...,أنَّه شَهِدَ النَّبيَّ صلَّى اللهُ عليه وسلَّم...,عبدالله بن زيد,ابن كثير,الأحكام الكبير,1/18,صحيح,أخرجه أحمد (16475)، وابن خزيمة (2931)، والبخار...
187397,72406,https://dorar.net/hadith/sharh/72406,أذان - بدء الأذان ، رؤيا - التواطؤ على الرؤيا ...,في هذا الحديث يحكي أبو عمير بن أنس بن مالك، عن...,اهتمَّ النبيُّ صلَّى اللهُ عليهِ وسلَّمَ للصلا...,عمومة أبي عمير بن أنس,الألباني,صحيح أبي داود,498,حسن,أخرجه أبو داود (498) واللفظ له، والبيهقي (1908...
